# Databricks Jobs

## O que são Jobs?

Jobs são a forma principal de orquestrar e agendar workloads no Databricks. Permitem executar notebooks, scripts Python, JARs, pipelines DLT e queries SQL de forma automatizada e programática.

**Para certificação:** Jobs são o mecanismo de **produção** para ETL/ELT — diferente de execução interactiva em notebooks.

---

## Anatomia de um Job

| Componente | Descrição |
|------------|-----------|
| **Job** | Container que agrupa uma ou mais tasks |
| **Task** | Unidade de trabalho individual |
| **Run** | Instância de execução de um job |
| **Trigger** | Condição que inicia o job |
| **Cluster** | Recurso computacional que executa as tasks |

---

## Tipos de Tasks

### Tasks Suportadas

| Tipo | Descrição | Caso de Uso |
|------|-----------|-------------|
| **Notebook** | Executa notebook Databricks | ETL, transformações, ML |
| **Python Script** | Executa ficheiro .py | Scripts standalone |
| **Python Wheel** | Executa pacote wheel | Bibliotecas empacotadas |
| **JAR** | Executa Java/Scala JAR | Aplicações Spark compiladas |
| **SQL** | Executa query ou dashboard | Agregações, relatórios |
| **DLT Pipeline** | Executa pipeline Delta Live Tables | Pipelines declarativos |
| **dbt** | Executa projecto dbt | Transformações dbt |
| **Run Job** | Executa outro job | Composição de jobs |
| **If/Else Condition** | Branching condicional | Lógica de fluxo |
| **For Each** | Loop sobre colecção | Processamento iterativo |

---

## Orquestração Multi-Task

### Dependências entre Tasks

Jobs suportam DAGs (Directed Acyclic Graphs) — tasks com dependências.

```
        ┌─────────────┐
        │   extract   │
        └──────┬──────┘
               │
        ┌──────▼──────┐
        │  transform  │
        └──────┬──────┘
               │
     ┌─────────┴─────────┐
     │                   │
┌────▼────┐        ┌─────▼─────┐
│  load   │        │  quality  │
└─────────┘        └───────────┘
```

### Configuração de Dependências

| Opção | Comportamento |
|-------|---------------|
| **Depends on** | Task executa após tasks especificadas |
| **Run if** | Condição para executar (All succeeded, At least one succeeded, None failed, All done) |

### Run If Conditions

| Condição | Descrição |
|----------|-----------|
| **All succeeded** | Todas as dependências com sucesso (padrão) |
| **At least one succeeded** | Pelo menos uma dependência com sucesso |
| **None failed** | Nenhuma dependência falhou (inclui skipped) |
| **All done** | Todas terminaram (independente do resultado) |

**Para certificação:** `None failed` permite que task execute mesmo se dependências foram skipped.

---

## Triggers (Agendamento)

### Tipos de Trigger

| Tipo | Descrição | Caso de Uso |
|------|-----------|-------------|
| **Manual** | Execução sob demanda | Testes, execuções pontuais |
| **Scheduled** | Cron ou intervalo fixo | Pipelines batch regulares |
| **Continuous** | Executa continuamente | Processamento near-real-time |
| **File Arrival** | Trigger quando ficheiros chegam | Ingestão event-driven |
| **Table Update** | Trigger quando tabelas Deltas são atualizadas | Pipelines ETL reativos, refresh de dashboards |

### Scheduled Trigger

#### Cron Expression

```
┌───────────── minuto (0-59)
│ ┌───────────── hora (0-23)
│ │ ┌───────────── dia do mês (1-31)
│ │ │ ┌───────────── mês (1-12)
│ │ │ │ ┌───────────── dia da semana (0-6, domingo=0)
│ │ │ │ │
* * * * *
```

| Expressão | Significado |
|-----------|-------------|
| `0 0 * * *` | Diariamente à meia-noite |
| `0 */2 * * *` | A cada 2 horas |
| `0 8 * * 1-5` | Dias úteis às 8h |
| `0 0 1 * *` | Primeiro dia de cada mês |

### Continuous Trigger

- Job reinicia automaticamente após conclusão
- Pausa configurável entre execuções
- Ideal para micro-batches ou polling

### File Arrival Trigger

```
Location: abfss://container@storage.dfs.core.windows.net/inbox/
Wait time: 5 minutes (tempo sem novos ficheiros para disparar)
```

**Comportamento:**
1. Monitoriza directório especificado
2. Aguarda período de "quiet time" sem novos ficheiros
3. Dispara job quando condição satisfeita

---

## Clusters para Jobs

### Job Cluster vs All-Purpose Cluster

| Aspecto | Job Cluster | All-Purpose Cluster |
|---------|-------------|---------------------|
| Ciclo de vida | Criado/destruído por run | Persistente |
| Custo | Menor (termina após job) | Maior (pode ficar idle) |
| Caso de uso | **Produção** | Desenvolvimento interactivo |
| Configuração | Definida no job | Independente |

**Para certificação:** Jobs em produção devem usar **Job Clusters** para optimização de custos.

### Cluster Reuse (Multi-Task)

| Opção | Comportamento |
|-------|---------------|
| **Shared Job Cluster** | Múltiplas tasks partilham o mesmo cluster |
| **Task-specific Cluster** | Cada task tem cluster dedicado |

**Vantagem do Shared:** Reduz tempo de startup entre tasks.

**Para certificação:** Tasks que partilham cluster executam **sequencialmente** no mesmo cluster.

---

## Retries e Timeouts

### Configuração de Retries

| Parâmetro | Descrição | Default |
|-----------|-----------|---------|
| **Max retries** | Número máximo de tentativas | 0 |
| **Min retry interval** | Tempo mínimo entre retries | 0 segundos |
| **Max retry interval** | Tempo máximo entre retries | 0 segundos |

### Timeout Configuration

| Nível | Parâmetro | Comportamento |
|-------|-----------|---------------|
| **Job** | Timeout | Cancela job inteiro |
| **Task** | Timeout | Cancela task específica |

**Para certificação:** Timeout de task **não cancela** outras tasks já em execução.

---

## Estados de Execução

### Job Run States

| Estado | Significado |
|--------|-------------|
| **PENDING** | Aguardando recursos |
| **RUNNING** | Em execução |
| **TERMINATING** | A terminar |
| **TERMINATED** | Concluído |
| **SKIPPED** | Não executou (condição não satisfeita) |
| **INTERNAL_ERROR** | Erro interno do Databricks |

### Result States

| Estado | Significado |
|--------|-------------|
| **SUCCESS** | Todas as tasks com sucesso |
| **FAILED** | Pelo menos uma task falhou |
| **TIMEDOUT** | Excedeu timeout |
| **CANCELED** | Cancelado manualmente |

---

## Repair Runs

### O que é Repair?

Permite re-executar **apenas tasks falhadas** de uma run anterior, sem re-executar tasks bem-sucedidas.

### Comportamento

| Aspecto | Descrição |
|---------|-----------|
| Escopo | Re-executa task falhada e dependentes |
| Task Values | Valores de tasks bem-sucedidas preservados |
| Cluster | Pode usar configuração diferente |
| Histórico | Mantém histórico da run original |

**Para certificação:** Repair é mais eficiente que re-executar job completo após falha parcial.

### Repair via UI

1. Abrir run falhada
2. Clicar "Repair run"
3. Seleccionar tasks a re-executar
4. Confirmar

---

## Permissões

### Níveis de Permissão

| Permissão | Ver | Executar | Editar | Gerir Permissões |
|-----------|-----|----------|--------|------------------|
| **No Permissions** | ❌ | ❌ | ❌ | ❌ |
| **Can View** | ✅ | ❌ | ❌ | ❌ |
| **Can Manage Run** | ✅ | ✅ | ❌ | ❌ |
| **Can Manage** | ✅ | ✅ | ✅ | ❌ |
| **Is Owner** | ✅ | ✅ | ✅ | ✅ |

### Run As (Identidade de Execução)

| Opção | Descrição |
|-------|-----------|
| **Owner** | Job executa com permissões do owner |
| **Service Principal** | Job executa com identidade de service principal |

**Para certificação:** Service Principals são recomendados para jobs de produção (não dependem de utilizador individual).

---

### Notificações

| Evento | Notificação |
|--------|-------------|
| Job Start | Email/webhook quando inicia |
| Job Success | Email/webhook em sucesso |
| Job Failure | Email/webhook em falha |
| Duration Warning | Quando excede duração esperada |

---

## Concurrency Control

### Concurrent Runs

| Configuração | Comportamento |
|--------------|---------------|
| **Max concurrent runs** | Número máximo de runs simultâneas |
| **Queue** | Runs extras aguardam na fila |
| **Skip** | Runs extras são descartadas |

**Para certificação:** Com `max_concurrent_runs = 1`, runs adicionais entram em fila ou são skipped conforme configuração.

## Referências Oficiais

- [Criar e Gerir Jobs](https://learn.microsoft.com/pt-pt/azure/databricks/workflows/jobs/create-run-jobs)
- [Configurar Clusters para Jobs](https://learn.microsoft.com/pt-pt/azure/databricks/workflows/jobs/jobs-compute)
- [Task Dependencies](https://learn.microsoft.com/pt-pt/azure/databricks/workflows/jobs/workflow-dependencies)
- [Repair Runs](https://learn.microsoft.com/pt-pt/azure/databricks/workflows/jobs/repair-job-failures)
- [Job Triggers](https://learn.microsoft.com/pt-pt/azure/databricks/workflows/jobs/triggers)